In [1]:
%pip install -q sentence-transformers

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [3]:
sentences = [
    "A neural network learns patterns from training data.",
    "Deep learning models learn representations from examples.",
    "The capital of France is Paris.",
    "Python is widely used for machine learning and AI development.",
    "A convolutional neural network is commonly used for image classification.",
    "Transformers use attention mechanisms to process relationships between tokens.",
    "The Eiffel Tower is a famous landmark in Paris.",
    "Semantic search retrieves text based on meaning rather than exact keywords.",
    "A database stores and organizes information for efficient retrieval.",
    "Gradient descent updates model parameters to reduce the loss."
]

embeddings = model.encode(sentences)

print("Embedding Shape: ", embeddings.shape)

Embedding Shape:  (10, 384)


In [4]:
query = "What does gradient descent do?"

query_embedding = model.encode(query)

print("Query embedding shape: ", query_embedding.shape)

Query embedding shape:  (384,)


In [5]:
from sentence_transformers import util

similarity_score = util.cos_sim(query_embedding, embeddings)

print("Similarity Score: ", similarity_score)

Similarity Score:  tensor([[0.3389, 0.3229, 0.0153, 0.1942, 0.2423, 0.2089, 0.0191, 0.1682, 0.1581,
         0.6401]])


In [6]:
scores = similarity_score[0]

top_k = 4
ranked_indices = scores.argsort(descending=True)[:top_k]

print("Query: ", query)
print("Top 4 results: -")

for idx in ranked_indices:
    i = idx.item()
    print(f"Score: {scores[i]:.4f} | {sentences[i]}")

Query:  What does gradient descent do?
Top 4 results: -
Score: 0.6401 | Gradient descent updates model parameters to reduce the loss.
Score: 0.3389 | A neural network learns patterns from training data.
Score: 0.3229 | Deep learning models learn representations from examples.
Score: 0.2423 | A convolutional neural network is commonly used for image classification.


In [33]:
documents = [
    {
        "title": "Gradient Descent",
        "text": """
        A machine learning model has parameters that change during training.
        A loss function measures how wrong the model's predictions are.
        Gradient descent calculates how changes to parameters affect the loss.
        It adjusts parameters using this information.
        The learning rate controls the size of each parameter update.
        """
    },
    {
        "title": "Neural Networks",
        "text": """
        Neural networks consist of interconnected layers of artificial
        neurons. During training, they learn patterns from examples by
        adjusting their parameters. Deep learning uses neural networks
        with multiple layers.
        """
    },
    {
        "title": "Transformers",
        "text": """
        Transformer models use attention mechanisms to represent relationships
        between tokens. They are widely used in language processing tasks,
        including text generation, translation, and semantic search.
        """
    }
]

for doc in documents:
    print(f"Title: {doc['title']}")
    print(doc["text"].strip())
    print()

Title: Gradient Descent
A machine learning model has parameters that change during training.
        A loss function measures how wrong the model's predictions are.
        Gradient descent calculates how changes to parameters affect the loss.
        It adjusts parameters using this information.
        The learning rate controls the size of each parameter update.

Title: Neural Networks
Neural networks consist of interconnected layers of artificial
        neurons. During training, they learn patterns from examples by
        adjusting their parameters. Deep learning uses neural networks
        with multiple layers.

Title: Transformers
Transformer models use attention mechanisms to represent relationships
        between tokens. They are widely used in language processing tasks,
        including text generation, translation, and semantic search.



In [8]:
from nltk.tokenize import sent_tokenize
import nltk

nltk.download("punkt_tab")

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [34]:
chunks = []

for doc in documents:
    sentences_in_doc = sent_tokenize(doc["text"])

    for sentence in sentences_in_doc:
        chunks.append({
            "title": doc["title"],
            "text": sentence
        })

for i, chunk in enumerate(chunks):
    print(f"Chunk {i + 1} | {chunk['title']}")
    print(chunk["text"])
    print()

Chunk 1 | Gradient Descent

        A machine learning model has parameters that change during training.

Chunk 2 | Gradient Descent
A loss function measures how wrong the model's predictions are.

Chunk 3 | Gradient Descent
Gradient descent calculates how changes to parameters affect the loss.

Chunk 4 | Gradient Descent
It adjusts parameters using this information.

Chunk 5 | Gradient Descent
The learning rate controls the size of each parameter update.

Chunk 6 | Neural Networks

        Neural networks consist of interconnected layers of artificial
        neurons.

Chunk 7 | Neural Networks
During training, they learn patterns from examples by
        adjusting their parameters.

Chunk 8 | Neural Networks
Deep learning uses neural networks
        with multiple layers.

Chunk 9 | Transformers

        Transformer models use attention mechanisms to represent relationships
        between tokens.

Chunk 10 | Transformers
They are widely used in language processing tasks,
        inc

In [35]:
import re

for chunk in chunks:
  chunk["text"] = re.sub(r"\s+", " ", chunk["text"]).strip()

In [36]:
chunk_texts = [chunk["text"] for chunk in chunks]

chunk_embeddings = model.encode(chunk_texts)

print("Number of chunks:", len(chunk_texts))
print("Embedding shape:", chunk_embeddings.shape)

Number of chunks: 10
Embedding shape: (10, 384)


In [12]:
doc_query = "How does the learning rate affect training?"

doc_query_embedding = model.encode(doc_query)

print("Query embedding: ", doc_query_embedding.shape)

Query embedding:  (384,)


In [13]:
doc_similarity_score = util.cos_sim(doc_query_embedding, chunk_embeddings)

print("Similarity Score: ", doc_similarity_score)

Similarity Score:  tensor([[0.3140, 0.2809, 0.7104, 0.1504, 0.4739, 0.2869, 0.1803, 0.1224]])


In [14]:
print(doc_similarity_score)

sorted_idx = doc_similarity_score[0].argsort(descending=True)

print("Query: ", doc_query)

for idx in sorted_idx:

  print(f"Score: {doc_similarity_score[0][idx]:.4f} | {chunk_texts[idx]}")


tensor([[0.3140, 0.2809, 0.7104, 0.1504, 0.4739, 0.2869, 0.1803, 0.1224]])
Query:  How does the learning rate affect training?
Score: 0.7104 | The learning rate controls the size of each update.
Score: 0.4739 | During training, they learn patterns from examples by adjusting their parameters.
Score: 0.3140 | Gradient descent is an optimization algorithm used to train machine learning models.
Score: 0.2869 | Deep learning uses neural networks with multiple layers.
Score: 0.2809 | It updates model parameters in the direction that reduces the loss function.
Score: 0.1803 | Transformer models use attention mechanisms to represent relationships between tokens.
Score: 0.1504 | Neural networks consist of interconnected layers of artificial neurons.
Score: 0.1224 | They are widely used in language processing tasks, including text generation, translation, and semantic search.


In [15]:
def search_document(query, top_k=4):

  query_embedding = model.encode(query)
  score = util.cos_sim(query_embedding, chunk_embeddings)

  top_k_idx = score[0].argsort(descending=True)[:top_k]

  result = []

  for i in top_k_idx:

    result.append({
        "title": chunks[i]["title"],
        "text": chunks[i]["text"],
        "score": score[0][i]
    })

  return result


In [16]:
res = search_document("What are transformers?")

print("Query: What are transformers?")

for r in res:
  print(f"Title: {r['title']}")
  print(f"Text: {r['text']}")
  print(f"Score: {r['score']}")

Query: What are transformers?
Title: Transformers
Text: Transformer models use attention mechanisms to represent relationships between tokens.
Score: 0.43603596091270447
Title: Transformers
Text: They are widely used in language processing tasks, including text generation, translation, and semantic search.
Score: 0.23822206258773804
Title: Neural Networks
Text: Deep learning uses neural networks with multiple layers.
Score: 0.19796153903007507
Title: Neural Networks
Text: During training, they learn patterns from examples by adjusting their parameters.
Score: 0.15039010345935822


### Text Generation

In [17]:
%pip install -q transformers sentencepiece

In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
generate_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model ready")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Model ready


In [19]:
# Perfoming tokenization

prompt = "Explain how gradient descent reduces the loss during training."

inputs = tokenizer(prompt, return_tensors="pt")

outputs = generate_model.generate(
    **inputs,
    max_new_tokens=100
)

print("Raw Output: ", outputs)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Answer: ", answer)

Raw Output:  tensor([[    0, 10771,  4741, 19991,    19,     3,     9,  1573,    13,     3,
         30819,     3,     9,  2412,     5,     1]])
Answer:  Gradient descent is a method of accelerating a train.


In [39]:
query = "How does gradient descent reduce loss?"

search_results = search_document(query, 3)

context = "\n".join(
    f"Source: {search_result['title']}\nPassage: {search_result['text']}" for search_result in search_results
)

prompt = f"""Answer the question in one concise sentence.
You can use the information supported by the context.
Do not repeat ideas and do not repeat the context word to word.

Context: {context}

Question: {query}
Answer:"""

inputs = tokenizer(prompt, return_tensors="pt")

outputs = generate_model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False,
)

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Query: ", query)
print("Retrived context: \n", context)
print("\nGenerated answer: \n", answer)

Query:  How does gradient descent reduce loss?
Retrived context: 
 Source: Gradient Descent
Passage: Gradient descent calculates how changes to parameters affect the loss.
Source: Gradient Descent
Passage: A loss function measures how wrong the model's predictions are.
Source: Neural Networks
Passage: Deep learning uses neural networks with multiple layers.

Generated answer: 
 Gradient descent calculates how changes to parameters affect the loss.


In [41]:
query = "How does gradient descent reduce loss?"

results = search_document(query, top_k=6)

for result in results:
    print(f"Score: {result['score']:.3f}")
    print(f"Source: {result['title']}")
    print(f"Passage: {result['text']}")
    print("-" * 40)

Score: 0.851
Source: Gradient Descent
Passage: Gradient descent calculates how changes to parameters affect the loss.
----------------------------------------
Score: 0.421
Source: Gradient Descent
Passage: A loss function measures how wrong the model's predictions are.
----------------------------------------
Score: 0.389
Source: Neural Networks
Passage: Deep learning uses neural networks with multiple layers.
----------------------------------------
Score: 0.334
Source: Gradient Descent
Passage: The learning rate controls the size of each parameter update.
----------------------------------------
Score: 0.314
Source: Gradient Descent
Passage: A machine learning model has parameters that change during training.
----------------------------------------
Score: 0.257
Source: Neural Networks
Passage: During training, they learn patterns from examples by adjusting their parameters.
----------------------------------------
